# S9 - Observabilidad con Grafana y costos
**Notebook:** `09_observabilidad_pipeline_grafana_costos` · **Entorno:** lambda26 (`pyspark`, red `lambda26-kafka-net`) · **Topic:** `atmos-eventos`

**Antes de ejecutar (fuera del notebook), deben estar vivos:**
1. `kafka`, `kafka-exporter` y `uso-atmos` arriba.
2. El **productor** corriendo en su propia terminal (`producer_sensores.py`) y el **consumidor** en otra (`consumer_sensores.py`).
3. `obs/` arriba (`docker compose -f obs/compose.yml up -d`) y pyspark unido a la red de Kafka.
4. Tablero de Grafana armado (pasos 3.2 y 3.3 se hacen en el navegador, no aquí).

Las celdas de Spark usan `startingOffsets = latest`: miden latencia sobre eventos **frescos**.

## Parámetros

In [1]:
TOPIC = "atmos-real"
BOOTSTRAP = "kafka:9092"
ARTIFACTS = "./artifacts/atmos_eventos_observabilidad"
CHECKPOINT_PARQUET = "./artifacts/chk_observabilidad"
TRIGGER = "5 seconds"

## 3.4 SparkSession con el conector de Kafka

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("sesion9-observabilidad")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
    .config("spark.sql.shuffle.partitions", "3")
    .config("spark.ui.port", "4040")
    .config("spark.sql.session.timeZone", "America/Lima")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4516efd7-7a53-447b-b5a0-4095294ea9ea;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9.2 in central
	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.mod

## 3.5 Lectura streaming y campos de observabilidad
`kafkaTimestamp` = cuándo el broker recibió el mensaje. `timestamp` = cuándo midió el sensor. `processedAt` = cuándo Spark procesó el evento. `latencyMs = processedAt - timestamp`.

In [3]:
from pyspark.sql.functions import col, from_json, current_timestamp, unix_millis
from pyspark.sql.types import StructType, StringType, DoubleType, LongType

esquema_evento = (
    StructType()
    .add("tipoEvento", StringType())
    .add("sensorId", StringType())
    .add("distanciaCm", DoubleType())
    .add("origen", StringType())
    .add("timestamp", LongType())
)

crudo = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "latest")
    .load()
)

eventos = (
    crudo.select(
        col("topic"), col("partition"), col("offset"),
        col("timestamp").alias("kafkaTimestamp"),
        col("value").cast("string").alias("json_str"),
    )
    .select("topic", "partition", "offset", "kafkaTimestamp",
            from_json(col("json_str"), esquema_evento).alias("e"))
    .select("topic", "partition", "offset", "kafkaTimestamp", "e.*")
)

observables = (
    eventos
    .withColumn("isValid",
            col("tipoEvento").isNotNull() & col("sensorId").isNotNull()
            & col("distanciaCm").isNotNull() & col("timestamp").isNotNull())
    .withColumn("processedAt", unix_millis(current_timestamp()))
    .withColumn("latencyMs", col("processedAt") - col("timestamp"))
)
print("es un stream:", observables.isStreaming)
observables.printSchema()

es un stream: True
root
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafkaTimestamp: timestamp (nullable = true)
 |-- tipoEvento: string (nullable = true)
 |-- sensorId: string (nullable = true)
 |-- distanciaCm: double (nullable = true)
 |-- origen: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- isValid: boolean (nullable = false)
 |-- processedAt: long (nullable = false)
 |-- latencyMs: long (nullable = true)



## 3.6 Consola con campos de observabilidad
En Jupyter el sink `console` imprime en el log del contenedor, no en la celda (hallazgo de S8). Por eso se arranca **además** un sink `memory` (`obs_mem`) para ver las filas con SQL en la propia celda. Las tres consultas siguen vivas hasta el final de 3.8.

In [4]:
consulta = (
    observables.writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", "false")
    .trigger(processingTime=TRIGGER)
    .start()
)

consulta_mem = (
    observables.writeStream
    .outputMode("append")
    .format("memory").queryName("obs_mem")
    .trigger(processingTime=TRIGGER)
    .start()
)
print("consulta activa:", consulta.isActive, "| consulta_mem activa:", consulta_mem.isActive)

consulta activa: True | consulta_mem activa: True
-------------------------------------------
Batch: 0
-------------------------------------------
+-----+---------+------+--------------+----------+--------+-----------+------+---------+-------+-----------+---------+
|topic|partition|offset|kafkaTimestamp|tipoEvento|sensorId|distanciaCm|origen|timestamp|isValid|processedAt|latencyMs|
+-----+---------+------+--------------+----------+--------+-----------+------+---------+-------+-----------+---------+
+-----+---------+------+--------------+----------+--------+-----------+------+---------+-------+-----------+---------+



[Stage 2:>                  (0 + 1) / 1][Stage 3:>                  (0 + 1) / 1]

-------------------------------------------
Batch: 1
-------------------------------------------
+----------+---------+------+-----------------------+--------------+----------------------+-----------+------------+-------------+-------+-------------+---------+
|topic     |partition|offset|kafkaTimestamp         |tipoEvento    |sensorId              |distanciaCm|origen      |timestamp    |isValid|processedAt  |latencyMs|
+----------+---------+------+-----------------------+--------------+----------------------+-----------+------------+-------------+-------+-------------+---------+
|atmos-real|0        |770   |2026-10-15 22:34:50.967|sensor.lectura|esp32-fisico-lanzeloth|119.0      |esp32-fisico|1792121690000|true   |1792121695006|5006     |
|atmos-real|0        |771   |2026-10-15 22:34:53.347|sensor.lectura|esp32-fisico-lanzeloth|118.9      |esp32-fisico|1792121693000|true   |1792121695006|2006     |
+----------+---------+------+-----------------------+--------------+--------------------

-------------------------------------------
Batch: 2
-------------------------------------------
+----------+---------+------+-----------------------+--------------+----------------------+-----------+------------+-------------+-------+-------------+---------+
|topic     |partition|offset|kafkaTimestamp         |tipoEvento    |sensorId              |distanciaCm|origen      |timestamp    |isValid|processedAt  |latencyMs|
+----------+---------+------+-----------------------+--------------+----------------------+-----------+------------+-------------+-------+-------------+---------+
|atmos-real|0        |772   |2026-10-15 22:34:56.352|sensor.lectura|esp32-fisico-lanzeloth|118.9      |esp32-fisico|1792121696000|true   |1792121700009|4009     |
|atmos-real|0        |773   |2026-10-15 22:34:59.346|sensor.lectura|esp32-fisico-lanzeloth|118.9      |esp32-fisico|1792121699000|true   |1792121700009|1009     |
+----------+---------+------+-----------------------+--------------+--------------------

In [7]:
# Esperar ~20 s y correr esta celda: eventos con sus campos de observabilidad
import time
time.sleep(20)
spark.sql("""
    SELECT sensorId, partition, offset, isValid, processedAt, latencyMs
    FROM obs_mem ORDER BY offset DESC
""").show(10, truncate=False)

+----------------------+---------+------+-------+-------------+---------+
|sensorId              |partition|offset|isValid|processedAt  |latencyMs|
+----------------------+---------+------+-------+-------------+---------+
|esp32-fisico-lanzeloth|0        |884   |true   |1792122035004|3004     |
|esp32-fisico-lanzeloth|0        |883   |true   |1792122030005|1005     |
|esp32-fisico-lanzeloth|0        |882   |true   |1792122030005|4005     |
|esp32-fisico-lanzeloth|0        |881   |true   |1792122025004|2004     |
|esp32-fisico-lanzeloth|0        |880   |true   |1792122025004|5004     |
|esp32-fisico-lanzeloth|0        |879   |true   |1792122020004|3004     |
|esp32-fisico-lanzeloth|0        |878   |true   |1792122015005|1005     |
|esp32-fisico-lanzeloth|0        |877   |true   |1792122015005|4005     |
|esp32-fisico-lanzeloth|0        |876   |true   |1792122010004|2004     |
|esp32-fisico-lanzeloth|0        |875   |true   |1792122010004|5004     |
+----------------------+---------+----

In [6]:
consulta.stop()
print("consulta de consola detenida | activas:", [q.name for q in spark.streams.active])

consulta de consola detenida | activas: ['obs_mem']


In [9]:
print("activa:", consulta_mem.isActive)
print("estado:", consulta_mem.status)
print("error:", consulta_mem.exception())
print("ultimo progreso:", consulta_mem.lastProgress)

activa: False
estado: {'message': 'Terminated with exception: Failed to create new KafkaAdminClient', 'isDataAvailable': False, 'isTriggerActive': False}
error: Failed to create new KafkaAdminClient
=== Streaming Query ===
Identifier: obs_mem [id = 2ea0fd9c-015b-4227-981d-ef37e7ccbfbc, runId = be2dedbd-bdf9-4711-9c03-c90c4240939a]
Current Committed Offsets: {}
Current Available Offsets: {}

Current State: ACTIVE
Thread State: RUNNABLE

Logical Plan:
WriteToMicroBatchDataSource MemorySink
+- ~Project [topic#9, partition#10, offset#11L, kafkaTimestamp#14, tipoEvento#17, sensorId#18, temperatura#19, humedad#20, presion#21, origen#22, timestamp#23L, isValid#24, processedAt#25L, (processedAt#25L - timestamp#23L) AS latencyMs#26L]
   +- ~Project [topic#9, partition#10, offset#11L, kafkaTimestamp#14, tipoEvento#17, sensorId#18, temperatura#19, humedad#20, presion#21, origen#22, timestamp#23L, isValid#24, unix_millis(current_timestamp()) AS processedAt#25L]
      +- ~Project [topic#9, partitio

## 3.7 Throughput y lag con `lastProgress`
`processedRowsPerSecond > inputRowsPerSecond` es la situación sana. `numInputRows = 0` no es error (lote `idle`).

In [8]:
import json

progreso = consulta.lastProgress
if progreso is None:
    print("Todavia no hay micro-lote procesado. Espera unos segundos y vuelve a correr esta celda.")
else:
    fuente = progreso["sources"][0]
    metricas = fuente.get("metrics", {})
    log = {
        "service": "spark-streaming",
        "component": "observabilidad",
        "topic": TOPIC,
        "batchId": progreso["batchId"],
        "numInputRows": progreso["numInputRows"],
        "inputRowsPerSecond": progreso["inputRowsPerSecond"],
        "processedRowsPerSecond": progreso["processedRowsPerSecond"],
        "avgOffsetsBehindLatest": metricas.get("avgOffsetsBehindLatest"),
        "maxOffsetsBehindLatest": metricas.get("maxOffsetsBehindLatest"),
        "startOffset": fuente.get("startOffset"),
        "endOffset": fuente.get("endOffset"),
        "status": "idle" if progreso["numInputRows"] == 0 else "processed",
    }
    print(json.dumps(log, indent=2))

{
  "service": "spark-streaming",
  "component": "observabilidad",
  "topic": "atmos-real",
  "batchId": 49,
  "numInputRows": 2,
  "inputRowsPerSecond": 0.4,
  "processedRowsPerSecond": 2.9,
  "avgOffsetsBehindLatest": "0.0",
  "maxOffsetsBehindLatest": "0",
  "startOffset": "{\"atmos-real\": {\"0\": 852, \"1\": 0, \"2\": 0}}",
  "endOffset": "{\"atmos-real\": {\"0\": 854, \"1\": 0, \"2\": 0}}",
  "status": "processed"
}


In [9]:
# Pico de throughput en los ultimos micro-lotes (insumo de la Tabla 9)
rp = consulta.recentProgress
tabla = [(p["batchId"], p["numInputRows"], p.get("inputRowsPerSecond"), p.get("processedRowsPerSecond")) for p in rp]
for t in tabla:
    print("batchId=%s filas=%s in/s=%s proc/s=%s" % t)
pico_in = max((t[2] or 0) for t in tabla) if tabla else None
print("\npico inputRowsPerSecond:", pico_in)

batchId=0 filas=0 in/s=0.0 proc/s=0.0
batchId=1 filas=2 in/s=0.4 proc/s=1.5
batchId=2 filas=2 in/s=0.4 proc/s=2.5
batchId=3 filas=1 in/s=0.2 proc/s=2.0
batchId=4 filas=2 in/s=0.4 proc/s=2.7
batchId=5 filas=2 in/s=0.4 proc/s=2.4
batchId=6 filas=2 in/s=0.4 proc/s=2.7
batchId=7 filas=1 in/s=0.2 proc/s=4.1
batchId=8 filas=3 in/s=0.6 proc/s=6.0
batchId=9 filas=2 in/s=0.4 proc/s=2.6
batchId=10 filas=1 in/s=0.2 proc/s=3.4
batchId=11 filas=2 in/s=0.4 proc/s=4.2
batchId=12 filas=2 in/s=0.4 proc/s=2.8
batchId=13 filas=1 in/s=0.2 proc/s=1.3
batchId=14 filas=1 in/s=0.2 proc/s=4.1
batchId=15 filas=3 in/s=0.6 proc/s=4.1
batchId=16 filas=2 in/s=0.4 proc/s=2.9
batchId=17 filas=1 in/s=0.2 proc/s=2.0
batchId=18 filas=2 in/s=0.4 proc/s=2.6
batchId=19 filas=2 in/s=0.4 proc/s=2.8
batchId=20 filas=1 in/s=0.2 proc/s=2.1
batchId=21 filas=2 in/s=0.4 proc/s=2.9
batchId=22 filas=2 in/s=0.4 proc/s=2.9
batchId=23 filas=1 in/s=0.2 proc/s=2.1
batchId=24 filas=2 in/s=0.4 proc/s=2.9
batchId=25 filas=2 in/s=0.4 proc/s=

**Evidencia alternativa:** Spark UI en http://localhost:4042 (puerto del host) → *Structured Streaming*.

## 3.8 Persistir en Parquet (en paralelo)
La segunda consulta se llama `consulta2` para no perder la referencia a `consulta`.

In [13]:
consulta2 = (
    observables.writeStream
    .outputMode("append")
    .format("parquet")
    .option("path", ARTIFACTS)
    .option("checkpointLocation", CHECKPOINT_PARQUET)
    .trigger(processingTime=TRIGGER)
    .start()
)
print("consulta2 activa:", consulta2.isActive)

consulta2 activa: True


In [12]:
ARTIFACTS = "./artifacts/atmos_real_observabilidad"
CHECKPOINT_PARQUET = "./artifacts/chk_real_observabilidad"
print(ARTIFACTS, CHECKPOINT_PARQUET)

./artifacts/atmos_real_observabilidad ./artifacts/chk_real_observabilidad


In [14]:
# Dejar correr 1-2 minutos. Repetir esta celda: el conteo debe crecer.
time.sleep(60)
guardado = spark.read.parquet(ARTIFACTS)
total = guardado.count()
print("filas guardadas:", total)
guardado.select("sensorId", "latencyMs", "isValid").show(10, truncate=False)

filas guardadas: 21
+----------------------+---------+-------+
|sensorId              |latencyMs|isValid|
+----------------------+---------+-------+
|esp32-fisico-lanzeloth|5003     |true   |
|esp32-fisico-lanzeloth|2003     |true   |
|esp32-fisico-lanzeloth|4003     |true   |
|esp32-fisico-lanzeloth|1003     |true   |
|esp32-fisico-lanzeloth|4003     |true   |
|esp32-fisico-lanzeloth|1003     |true   |
|esp32-fisico-lanzeloth|6004     |true   |
|esp32-fisico-lanzeloth|3004     |true   |
|esp32-fisico-lanzeloth|5003     |true   |
|esp32-fisico-lanzeloth|2003     |true   |
+----------------------+---------+-------+
only showing top 10 rows


In [15]:
# Muestra 1 para crecimiento del Parquet (Tabla 9). Anota hora y total.
import datetime, os
def tam_mb(ruta):
    t = 0
    for d, _, fs in os.walk(ruta):
        for f in fs:
            if f.endswith(".parquet"):
                t += os.path.getsize(os.path.join(d, f))
    return t / 1e6
m1 = (datetime.datetime.now(), spark.read.parquet(ARTIFACTS).count(), tam_mb(ARTIFACTS))
print("muestra 1:", m1)

muestra 1: (datetime.datetime(2026, 10, 16, 3, 49, 46, 596431), 65, 0.13939)


In [18]:
consulta.stop()
print("activas:", [q.name or q.id for q in spark.streams.active])
print("m1:", m1)

activas: ['692d90de-c32b-44e4-ab5c-e5f406e28643', 'obs_mem']
m1: (datetime.datetime(2026, 10, 9, 20, 12, 58, 874349), 216, 0.377178)


## Detener las consultas (cuando ya tengas lo necesario, tras la muestra 2 de 3.11)

In [ ]:
# NO correr todavia si quieres tomar la muestra 2 de crecimiento (celda de 3.11).
# consulta.stop(); consulta_mem.stop(); consulta2.stop()
print("activas:", [q.name or q.id for q in spark.streams.active])

## 3.9 Latencia agregada por minuto (batch sobre el Parquet)
`minLatencyMs`/`maxLatencyMs` varían porque el productor emite cada 3 s y el trigger de Spark es de 5 s (mcm = 15 s): la espera cicla entre ~0 y ~5 s. No es una falla.

In [16]:
from pyspark.sql.functions import window, avg, count, expr
from pyspark.sql.functions import min as spark_min, max as spark_max

guardado = spark.read.parquet(ARTIFACTS)

latencia_por_minuto = (
    guardado.groupBy(window(col("kafkaTimestamp"), "1 minute"))
    .agg(
        count("*").alias("numEventos"),
        avg("latencyMs").alias("avgLatencyMs"),
        spark_min("latencyMs").alias("minLatencyMs"),
        spark_max("latencyMs").alias("maxLatencyMs"),
        expr("percentile_approx(latencyMs, 0.95)").alias("p95LatencyMs"),
    )
    .select(col("window.start").alias("inicioVentana"), col("window.end").alias("finVentana"),
            "numEventos", "avgLatencyMs", "minLatencyMs", "maxLatencyMs", "p95LatencyMs")
    .orderBy("inicioVentana")
)
latencia_por_minuto.show(20, truncate=False)

+-------------------+-------------------+----------+------------+------------+------------+------------+
|inicioVentana      |finVentana         |numEventos|avgLatencyMs|minLatencyMs|maxLatencyMs|p95LatencyMs|
+-------------------+-------------------+----------+------------+------------+------------+------------+
|2026-10-15 22:46:00|2026-10-15 22:47:00|10        |3003.2      |1003        |5003        |5003        |
|2026-10-15 22:47:00|2026-10-15 22:48:00|20        |3503.85     |1003        |7004        |6004        |
|2026-10-15 22:48:00|2026-10-15 22:49:00|20        |4754.5      |1003        |16004       |13004       |
|2026-10-15 22:49:00|2026-10-15 22:50:00|20        |3253.4      |1003        |8003        |5004        |
|2026-10-15 22:50:00|2026-10-15 22:51:00|20        |3503.2      |1002        |8004        |6003        |
|2026-10-15 22:51:00|2026-10-15 22:52:00|13        |3080.0      |1003        |5003        |5003        |
+-------------------+-------------------+----------+---

In [17]:
# Resumen global (insumo de los umbrales de 3.10 y de la Tabla 9)
resumen = guardado.agg(
    count("*").alias("n"),
    avg("latencyMs").alias("avg_ms"),
    spark_max("latencyMs").alias("max_ms"),
    expr("percentile_approx(latencyMs, 0.95)").alias("p95_ms"),
    expr("percentile_approx(latencyMs, 0.99)").alias("p99_ms"),
).collect()[0]
print(resumen)
AVG_MS, P95_MS, MAX_MS = float(resumen["avg_ms"]), float(resumen["p95_ms"]), float(resumen["max_ms"])
print("\nLatencia promedio (ms):", round(AVG_MS, 1), "| p95:", P95_MS, "| max:", MAX_MS)

Row(n=103, avg_ms=3595.8349514563106, max_ms=16004, p95_ms=7004, p99_ms=13004)

Latencia promedio (ms): 3595.8 | p95: 7004.0 | max: 16004.0


## 3.10 Alertas propuestas y umbrales
Los umbrales de latencia se contrastan con **lo medido** arriba, no con valores supuestos.

In [21]:
import pandas as pd
alertas = pd.DataFrame([
    ("Kafka no visible",       "kafka_brokers < 1",                                   "Prometheus / Grafana", "1 si todo va bien (medido en 3.2)"),
    ("Exporter caído",         'up{job="kafka-exporter"} == 0',                       "Prometheus / Grafana", "1 si todo va bien (medido en 3.2)"),
    ("Lag alto sostenido",     "kafka_consumergroup_lag > 10 (2 min)",                "Prometheus / Grafana", "lag normal medido: 0 a 1 (avgOffsetsBehindLatest = 0.0)"),
    ("Latencia alta sensible", "avgLatencyMs > 4000 (por minuto, 3 min seguidos)",    "Spark / Parquet",      f"promedio medido: {AVG_MS:.0f} ms, p95: {P95_MS:.0f} ms"),
    ("Latencia alta crítica",  "p95LatencyMs > 12000 (por minuto)",                   "Spark / Parquet",      f"máximo medido: {MAX_MS:.0f} ms (ventana 22:48, p95 13004 ms)"),
], columns=["Situacion", "Regla", "Donde se evalua", "Dato medido que la justifica"])
alertas

,Situacion,Regla,Donde se evalua,Dato medido que la justifica
0,Kafka no visible,kafka_brokers < 1,Prometheus / Grafana,1 si todo va bien (medido en 3.2)
1,Exporter caído,"up{job=""kafka-exporter""} == 0",Prometheus / Grafana,1 si todo va bien (medido en 3.2)
2,Lag alto sostenido,kafka_consumergroup_lag > 10 (2 min),Prometheus / Grafana,lag normal medido: 0 a 1 (avgOffsetsBehindLate...
3,Latencia alta sensible,"avgLatencyMs > 4000 (por minuto, 3 min seguidos)",Spark / Parquet,"promedio medido: 3596 ms, p95: 7004 ms"
4,Latencia alta crítica,p95LatencyMs > 12000 (por minuto),Spark / Parquet,"máximo medido: 16004 ms (ventana 22:48, p95 13..."


**Lectura de los umbrales:** con datos reales del ESP32 (103 eventos, 22:46 a 22:52) la latencia fue 3596 ms de promedio, p95 7004 ms y máximo 16004 ms. La latencia normal de 1 a 8 s viene del ciclo productor 3 s contra trigger 5 s y de la resolución de 1 s del timestamp del ESP32. Por eso umbrales de 100 o 1000 ms darían alarma permanente. Se propone avgLatencyMs > 4000 ms durante 3 minutos seguidos (sensible) y p95LatencyMs > 12000 ms por minuto (crítica). En la corrida, la ventana 22:48 llegó a avg 4754 ms, p95 13004 ms y máx. 16004 ms. En Grafana, la tasa de mensajes cae y luego sube de golpe hacia las 22:49–22:50, lo que apunta a una demora aguas arriba (WiFi/MQTT/puente) más que a Spark. Dispara la alerta crítica una vez y la sensible no, porque no se sostuvo 3 minutos.

## 3.11 Estimación de costos y plan de escalado
Precios de referencia de la guía (AWS us-east-1, oct-2026; **verifícalos antes de entregar**): t3.medium ≈ US\$0.0416/h; S3 Standard ≈ US\$0.023/GB-mes.

In [19]:
# Muestra 2 de crecimiento: correr >= 5 min despues de la muestra 1 (cuanto mas tiempo, mejor)
m2 = (datetime.datetime.now(), spark.read.parquet(ARTIFACTS).count(), tam_mb(ARTIFACTS))
horas = (m2[0] - m1[0]).total_seconds() / 3600
filas_h = (m2[1] - m1[1]) / horas
mb_h = (m2[2] - m1[2]) / horas
print("muestra 1:", m1)
print("muestra 2:", m2)
print(f"crecimiento: {filas_h:,.0f} filas/hora | {mb_h:.2f} MB/hora")

muestra 1: (datetime.datetime(2026, 10, 16, 3, 49, 46, 596431), 65, 0.13939)
muestra 2: (datetime.datetime(2026, 10, 16, 3, 54, 15, 691309), 153, 0.337559)
crecimiento: 1,177 filas/hora | 2.65 MB/hora


In [22]:
PRECIO_T3 = 0.0416      # US$/hora, t3.medium bajo demanda
PRECIO_S3 = 0.023       # US$/GB-mes
HORAS_MES = 730
INSTANCIAS = 3          # Kafka(+exporter), streaming, obs (Prometheus+Grafana)

cap_in = pico_in if pico_in else 0
gb_mes = mb_h * HORAS_MES / 1000
costo_computo = INSTANCIAS * PRECIO_T3 * HORAS_MES
costo_alm = gb_mes * PRECIO_S3

tabla9 = pd.DataFrame([
    ("Eventos por segundo (pico)",        f"{cap_in:.2f} (lastProgress)",             "1 sensor ESP32 cada 3 s = 0.33 evento/s"),
    ("Latencia promedio por minuto",      f"{AVG_MS:.0f} ms (p95 {P95_MS:.0f} ms)",   "ciclo 3 s productor vs 5 s trigger"),
    ("Crecimiento del Parquet por hora",  f"{mb_h:.2f} MB ({filas_h:,.0f} filas)",    f"= {gb_mes:.3f} GB/mes en S3"),
    ("Computo por hora (3 instancias)",   f"US$ {INSTANCIAS*PRECIO_T3:.4f}",          "3 x t3.medium on-demand"),
    ("Computo mensual",                   f"US$ {costo_computo:.2f}",                 "x 730 h"),
    ("Almacenamiento mensual",            f"US$ {costo_alm:.3f}",                     "S3 Standard"),
    ("TOTAL base mensual",                f"US$ {costo_computo + costo_alm:.2f}",     "sin red ni mano de obra"),
], columns=["Variable", "Valor medido / calculado", "Supuesto"])
tabla9

,Variable,Valor medido / calculado,Supuesto
0,Eventos por segundo (pico),0.60 (lastProgress),1 sensor ESP32 cada 3 s = 0.33 evento/s
1,Latencia promedio por minuto,3596 ms (p95 7004 ms),ciclo 3 s productor vs 5 s trigger
2,Crecimiento del Parquet por hora,"2.65 MB (1,177 filas)",= 1.935 GB/mes en S3
3,Computo por hora (3 instancias),US$ 0.1248,3 x t3.medium on-demand
4,Computo mensual,US$ 91.10,x 730 h
5,Almacenamiento mensual,US$ 0.045,S3 Standard
6,TOTAL base mensual,US$ 91.15,sin red ni mano de obra


### Plan de escalado (con números medidos en esta corrida)

**Punto de partida medido:** 1 sensor ESP32 físico publicando cada 3 s (≈ 0.33 evento/s; pico de lastProgress 0.6 filas/s), latencia promedio 3596 ms y p95 7004 ms, lag 0, Parquet creciendo ≈ 2.65 MB/h (1 177 filas/h) y costo base ≈ US\$ 91.15/mes (3 × t3.medium + S3).

**Cuellos de botella detectados hoy**
- *Una sola partición activa:* el productor usa `sensorId` como key y hay un solo sensor, así que todo cae en la partición 0 (las particiones 1 y 2 están vacías y Prometheus reporta lag -1 en ellas). Tener 3 particiones no da paralelismo hasta que haya más sensores o una key distinta.
- *Un solo broker con replicación 1:* si Kafka cae, se pierden lecturas. No es tolerante a fallos.
- *Latencia dominada por el trigger:* 1 a 8 s por el ciclo de 3 s (productor) contra 5 s (trigger) y por el timestamp de 1 s del ESP32; no por falta de capacidad.
- *Archivos Parquet pequeños:* un archivo por micro-lote; el tamaño real por hora está inflado por ese sobrecoste.

**Escenarios**
| Escenario | Eventos/s | Qué cambia | Costo adicional estimado |
|---|---|---|---|
| Hoy (1 sensor) | 0.33 | Nada | US\$ 91.15/mes |
| 100 sensores | ≈ 33 | Keys distintas reparten las 3 particiones; subir el trigger a 10 s o más; compactar Parquet | Parquet ≈ 193 GB/mes como cota alta ≈ US\$ 4.5/mes en S3 |
| 1 000 sensores | ≈ 333 | 3 brokers (replicación 3), 6 a 12 particiones, Spark con varios executors (cluster), Prometheus con retención y almacenamiento remoto | Más instancias de cómputo; el cómputo domina el costo |

**Alertas que acompañan al escalado:** lag > 10 durante 2 min (consumidor atrasado), latencia p95 por minuto > 12 000 ms y `up{job="kafka-exporter"} == 0`.

*Los escenarios de 100 y 1 000 sensores son extrapolaciones lineales de lo medido, no pruebas de carga reales.*

## Cierre

In [23]:
for q in spark.streams.active:
    q.stop()
print("consultas activas:", len(spark.streams.active))

consultas activas: 0


26/10/16 04:05:37 ERROR FileFormatWriter: Aborting job e71f9994-d84e-40d4-bf01-dd644031a3a3.
java.lang.InterruptedException
	at java.base/java.util.concurrent.locks.AbstractQueuedSynchronizer.tryAcquireSharedNanos(AbstractQueuedSynchronizer.java:1167)
	at scala.concurrent.impl.Promise$DefaultPromise.tryAwait0(Promise.scala:251)
	at scala.concurrent.impl.Promise$DefaultPromise.result(Promise.scala:271)
	at org.apache.spark.util.SparkThreadUtils$.awaitResultNoSparkExceptionConversion(SparkThreadUtils.scala:78)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.B